# Analise de Coordenadas entre CSVs

No notebook anterior, a gente viu que existem 1.351 candidatos positivos em `candidates.csv` e 1.186 anotacoes em `annotations.csv`. Mas as coordenadas entre os dois arquivos nao batem exatamente. Por exemplo:

- **annotations.csv**: `(-128.70, -175.32, -298.39)`
- **candidates.csv**: `(-128.94, -175.04, -297.87)`

Ambas representam o mesmo nodulo (~5mm de diametro), mas com ~0.5mm de diferenca. Neste notebook, a gente vai medir essa diferenca pra saber qual threshold usar na hora de unificar as duas fontes.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = Path("../data/luna")

## Medindo o desalinhamento

Pra cada candidato positivo, vamos achar a anotacao mais proxima (do mesmo CT) e calcular a distancia euclidiana entre as duas coordenadas.

In [ ]:
annotations_df = pd.read_csv(DATA_DIR / "annotations.csv")
candidates_df = pd.read_csv(DATA_DIR / "candidates.csv")

print(f"Anotacoes: {len(annotations_df)}")
print(f"Candidatos: {len(candidates_df)}")
print(f"Candidatos positivos: {candidates_df['class'].sum()}")

In [ ]:
positive_candidates = candidates_df[candidates_df["class"] == 1].copy()

distances = []
matched_diameters = []

for _, cand in positive_candidates.iterrows():
    ann_for_series = annotations_df[
        annotations_df["seriesuid"] == cand["seriesuid"]
    ]
    if len(ann_for_series) == 0:
        continue

    min_dist = float("inf")
    matched_diam = 0

    for _, ann in ann_for_series.iterrows():
        dist = np.sqrt(
            (cand["coordX"] - ann["coordX"]) ** 2
            + (cand["coordY"] - ann["coordY"]) ** 2
            + (cand["coordZ"] - ann["coordZ"]) ** 2
        )
        if dist < min_dist:
            min_dist = dist
            matched_diam = ann["diameter_mm"]

    distances.append(min_dist)
    matched_diameters.append(matched_diam)

distances = np.array(distances)
matched_diameters = np.array(matched_diameters)
print(f"Candidatos positivos analisados: {len(distances)}")

In [ ]:
print("Distancia entre candidato e anotacao mais proxima (mm):")
print(f"  Min:     {distances.min():.3f}")
print(f"  Max:     {distances.max():.3f}")
print(f"  Media:   {distances.mean():.3f}")
print(f"  Mediana: {np.median(distances):.3f}")
print(f"  Desvio:  {distances.std():.3f}")

A diferenca media e de ~2.4mm, com o pior caso chegando a ~16mm. Vamos visualizar isso pra entender melhor:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(distances, bins=50, color="steelblue", edgecolor="white", alpha=0.7)
axes[0].axvline(distances.mean(), color="red", linestyle="--",
                label=f"Media: {distances.mean():.2f}mm")
axes[0].axvline(np.median(distances), color="orange", linestyle="--",
                label=f"Mediana: {np.median(distances):.2f}mm")
axes[0].set_xlabel("Distancia (mm)")
axes[0].set_ylabel("Quantidade")
axes[0].set_title("Distancia candidato vs anotacao")
axes[0].legend()

axes[1].scatter(matched_diameters, distances, alpha=0.5, s=20)
x_range = np.linspace(3, 35, 100)
axes[1].plot(x_range, x_range / 2, "r--", label="Raio (diametro/2)")
axes[1].set_xlabel("Diametro do nodulo (mm)")
axes[1].set_ylabel("Distancia (mm)")
axes[1].set_title("Desalinhamento vs tamanho do nodulo")
axes[1].legend()

plt.tight_layout()
plt.show()

Olha so o grafico da direita: todos os pontos ficam abaixo da linha vermelha (`diametro / 2`). Isso quer dizer que o desalinhamento e sempre menor que o raio do nodulo. Bom sinal -- da pra fazer o matching com seguranca.

Vamos confirmar com numeros:

In [ ]:
within_radius = (distances <= matched_diameters / 2).sum()
print(f"Dentro do raio (diametro/2): {within_radius}/{len(distances)} ({100*within_radius/len(distances):.1f}%)")

for factor in [0.6, 0.7, 0.8, 1.0]:
    within = (distances <= matched_diameters * factor).sum()
    print(f"  Dentro de diametro*{factor}: {within}/{len(distances)} ({100*within/len(distances):.1f}%)")

100% dos candidatos positivos ficam dentro do raio (`diametro/2`). No proximo notebook, a gente usa `diametro/4` por eixo (que e mais conservador) pra fazer o matching na pratica. Essa margem garante que a gente nao perde nenhum nodulo na hora de unificar os CSVs.